In [ ]:
import sys
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

# Minimal environment setup to keep the notebook focused on the core experiments.
# If you run this elsewhere, update PROJECT_DIR accordingly.
PROJECT_DIR = Path("/content/drive/MyDrive/0.Portfolio/electricity_price_forecasting").resolve()


In [ ]:
project_str = str(PROJECT_DIR)
if project_str not in sys.path:
    sys.path.insert(0, project_str)

DATA_DIR         = PROJECT_DIR / "data"
RESULTS_DIR      = PROJECT_DIR / "results"
METRICS_DIR      = RESULTS_DIR / "metrics"
MODELS_DIR       = RESULTS_DIR / "models"
WINDOW_INDEX_DIR = RESULTS_DIR / "window_index"
LOGS_DIR         = RESULTS_DIR / "logs"
CACHE_DIR        = RESULTS_DIR / "cache"
PLOTS_DIR        = RESULTS_DIR / "plots"

for p in [
    RESULTS_DIR,
    PLOTS_DIR,
    METRICS_DIR,
    MODELS_DIR,
    WINDOW_INDEX_DIR,
    LOGS_DIR,
    CACHE_DIR,
    PLOTS_DIR / "models",
    PLOTS_DIR / "eval/curves",
    PLOTS_DIR / "eval/tables",
    PLOTS_DIR / "eda",
    PLOTS_DIR / "feature_engineering",
]:
    p.mkdir(parents=True, exist_ok=True)


# 00_Demo

Baseline + data sanity + shared constants.


In [ ]:
# Constants
try:
    from electricity_forecasting.config import (
        WIN, HORIZON,
        MAX_TEST_WINDOWS, MAX_TRAIN_WINDOWS, MAX_VAL_WINDOWS,
        VALID_RATIO,
        TARGET_COLS, VOLUME_INDEX,
        FREQ, BERLIN_TZ,
    )
except ModuleNotFoundError:
    import sys
    from pathlib import Path
    src_dir = Path(PROJECT_DIR) / "src"
    if str(src_dir) not in sys.path:
        sys.path.insert(0, str(src_dir))
    from electricity_forecasting.config import (
        WIN, HORIZON,
        MAX_TEST_WINDOWS, MAX_TRAIN_WINDOWS, MAX_VAL_WINDOWS,
        VALID_RATIO,
        TARGET_COLS, VOLUME_INDEX,
        FREQ, BERLIN_TZ,
    )

print("WIN:", WIN)
print("HORIZON:", HORIZON)
print("MAX_TEST_WINDOWS:", MAX_TEST_WINDOWS)


In [ ]:
# Parquet paths
TRAIN_PARQUET = DATA_DIR / "TRAIN_Reco_2021_2022_2023_with_time_utc.parquet"
TEST_PARQUET  = DATA_DIR / "TEST_Reco_2024_with_time_utc.parquet"


In [ ]:
# ID mapping (stable codes)
from electricity_forecasting.io.id_mapping import (
    scan_unique_ids_from_parquet,
    build_id_mapping,
    save_id_mapping_json,
)

# 1) Scan unique IDs from parquet (RAM-safe)
ids_raw = scan_unique_ids_from_parquet(str(TEST_PARQUET), id_col="ID")

# 2) Build mapping
id_to_code, code_to_id, codes = build_id_mapping(ids_raw)

# 3) Save mapping JSON
out_map = RESULTS_DIR / "cache" / "id_code_mapping.json"
save_id_mapping_json(out_map, id_to_code=id_to_code, code_to_id=code_to_id)

print("n_ids_test:", len(codes))

n_ids_test: 672


In [ ]:
# Build / persist window indices (ID-balanced, deterministic)
from electricity_forecasting.datasets.window_index_builder import build_window_indices
# NOTE: build_window_indices expects all_id_codes (not all_codes)
from electricity_forecasting.evaluation.evaluate import save_window_index, save_json

train_start = pd.Timestamp("2021-01-01 00:00:00", tz="UTC")
train_end   = pd.Timestamp("2023-12-31 23:45:00", tz="UTC")
test_start  = pd.Timestamp("2024-01-01 00:00:00", tz="UTC")
test_end    = pd.Timestamp("2024-12-31 23:45:00", tz="UTC")

w_tr, w_va, w_te, wi_meta = build_window_indices(
    all_id_codes=codes,
    train_start_utc=train_start,
    train_end_utc=train_end,
    test_start_utc=test_start,
    test_end_utc=test_end,
    win=WIN,
    horizon=HORIZON,
    freq=FREQ,
    max_train_windows=MAX_TRAIN_WINDOWS,
    max_val_windows=MAX_VAL_WINDOWS,
    max_test_windows=MAX_TEST_WINDOWS,
    valid_ratio=VALID_RATIO,
    seed=42,
)

save_window_index(w_tr, RESULTS_DIR / "window_index" / "train_windows_all_ids_cap300000.csv")
save_window_index(w_va, RESULTS_DIR / "window_index" / "val_windows_all_ids_cap100000.csv")
save_window_index(w_te, RESULTS_DIR / "window_index" / "test_windows_all_ids_cap200000.csv")

# Save meta for reproducibility (optional)
save_json(wi_meta, RESULTS_DIR / "window_index" / "window_index_meta.json")

print("n_train_windows:", len(w_tr))
print("n_val_windows:", len(w_va))
print("n_test_windows:", len(w_te))


TypeError: build_window_indices() got an unexpected keyword argument 'all_codes'

In [ ]:
# Baseline evaluation (persistence) on test windows
from electricity_forecasting.datasets.materialize import materialize_from_window_index
from electricity_forecasting.evaluation.evaluate import (
    evaluate_predictions,
    build_metrics_record,
    save_metrics_record,
    active_ratio_from_truth,
    log_run_header,
)
from electricity_forecasting.evaluation.metrics_table_update import update_metrics_table

# Materialize tensors from window_index
X_te, Y_te, Yb_te, yv_te, ids_te, feature_cols_used = materialize_from_window_index(
    parquet_path=str(TEST_PARQUET),
    window_index=w_te,
    code_to_id=code_to_id,
    start_utc=test_start,
    end_utc=test_end,
    freq=FREQ,
    berlin_tz=BERLIN_TZ,
    win=WIN,
    horizon=HORIZON,
    target_cols=TARGET_COLS,
    volume_index=VOLUME_INDEX,
    feature_cols=None,
)

# Shapes (required reproducibility check)
shapes = {
    "X_test": list(X_te.shape),
    "Y_test": list(Y_te.shape),
    "Yb_test": list(Yb_te.shape),
}
sampling_info = {
    "n_ids_test": int(len(codes)),
    "n_test_windows_used": int(Y_te.shape[0]),
    "sampling_rule": "ID-balanced, deterministic ordering",
    "window_index_file": "results/window_index/test_windows_all_ids_cap200000.csv",
}

log_run_header(
    model_name="baseline_persistence",
    constants={
        "WIN": WIN,
        "HORIZON": HORIZON,
        "MAX_TEST_WINDOWS": MAX_TEST_WINDOWS,
        "FREQ": FREQ,
        "TARGET_COLS": list(TARGET_COLS),
        "VOLUME_INDEX": int(VOLUME_INDEX),
    },
    shapes=shapes,
    sampling_info=sampling_info,
    threshold_info={"threshold_selected_on": "validation"},
)

# Baseline metrics (Yb_te is the persistence baseline)
baseline_metrics = evaluate_predictions(Y_true=Y_te, Y_pred=Yb_te, volume_index=VOLUME_INDEX)

# In the demo notebook, "model" == baseline (so the table schema stays consistent)
model_metrics = dict(baseline_metrics)

active_ratio_test = active_ratio_from_truth(Y_te, volume_index=VOLUME_INDEX)

record = build_metrics_record(
    model_name="baseline_persistence",
    run_tag="demo",
    constants={
        "WIN": WIN,
        "HORIZON": HORIZON,
        "MAX_TEST_WINDOWS": MAX_TEST_WINDOWS,
        "FREQ": FREQ,
        "TARGET_COLS": list(TARGET_COLS),
        "VOLUME_INDEX": int(VOLUME_INDEX),
    },
    split_info={
        "train_years": "2021-2023",
        "test_year": "2024",
        "valid_ratio": float(VALID_RATIO),
    },
    sampling_info=sampling_info,
    baseline_metrics=baseline_metrics,
    model_metrics=model_metrics,
    active_ratio_test=float(active_ratio_test),
    threshold_info={"threshold_selected_on": "validation"},
)

out_json = RESULTS_DIR / "metrics" / "baseline_persistence.json"
save_metrics_record(record, out_json)
print("Saved metrics JSON:", out_json.name)

# Update shared metrics table (CSV/XLSX/PNG)
df = update_metrics_table(
    metrics_dir=RESULTS_DIR / "metrics",
    out_csv=RESULTS_DIR / "metrics" / "metrics_table.csv",
    out_xlsx=RESULTS_DIR / "metrics" / "metrics_table.xlsx",
    out_png=PLOTS_DIR / "eval/tables" / "metrics_table.png",
)
display(df)
